# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import numpy as np
import pandas as pd
import duckdb
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

EARLY_START = "2026-03-01"
EARLY_END = "2026-03-07"

RECENT_START = "2026-03-09"
RECENT_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

print("sklearn:", sklearn.__version__)
print("Setup complete.")

AssertionError: HF_TOKEN not found.

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

**Paper finding:** The paper reports that mature 365+ day content that had been
refreshed within 30 days showed approximately 3.2× higher health score and 57×
more impressions than older content in the comparison group.

**Methodology question:** How comparable were the refreshed and unrefreshed
mature-page groups before the refresh occurred?

In particular, I would want to know whether the performance window occurs
strictly after the refresh event and whether pages selected for refreshing
already differed in historical visibility, topic demand, content quality, or
editorial priority. If stronger pages were more likely to be selected for
refresh, some of the measured difference could reflect selection effects rather
than the refresh itself.

The finding is therefore useful as an observed portfolio association, but I
would be cautious about interpreting the 3.2× and 57× differences as causal
effects without a matched, longitudinal, or experimental comparison.

### Finding 2 — Logistic Regression: What Predicts Growth?

**Paper finding:** The exploratory ML appendix reports 71% holdout accuracy for
a Logistic Regression model separating growing from declining pages. Content
age is described as the strongest negative signal, while days visible and
recent impressions are among the stronger positive signals.

**Methodology question:** How exactly was the growing-versus-declining label
constructed, and were all model features measured strictly before the period
used to define that label?

The paper defines trend direction using a 30-day-versus-previous-30-day
impression comparison. I would therefore check whether features such as recent
impressions or days visible overlap either window used to construct the target.

I would also ask whether the reported 80/20 holdout was page-random,
client-grouped, or time-aware. If pages from the same brand occur in both train
and test sets, the model may benefit from shared client-specific characteristics
and the measured performance may overstate generalization to an unseen client.

Finally, I would report the positive-class base rate next to the 71% accuracy,
because accuracy alone does not show how much improvement exists over a simple
majority-class prediction.

In [2]:
import pandas as pd
from IPython.display import display

In [3]:
paper_audit = pd.DataFrame([
    {
        "finding": "Freshness Multiplier",
        "paper_page": 9,
        "reported_result": "3.2x health; 57x impressions",
        "main_audit_question": "Are refreshed and unrefreshed mature pages comparable?"
    },
    {
        "finding": "What Predicts Growth?",
        "paper_page": 29,
        "reported_result": "71% Logistic Regression holdout accuracy",
        "main_audit_question": "Is the label temporally separated and is validation grouped?"
    }
])

display(paper_audit)


,finding,paper_page,reported_result,main_audit_question
0,Freshness Multiplier,9,3.2x health; 57x impressions,Are refreshed and unrefreshed mature pages com...
1,What Predicts Growth?,29,71% Logistic Regression holdout accuracy,Is the label temporally separated and is valid...


## 2. My model under an honest split (before/after)

My Week-5 submission already used a client-grouped holdout. To make the effect
of validation design visible in this audit, I reconstruct a weaker page-random
80/20 split as the "before" condition and compare it with the client-grouped
80/20 split used in Week 5.

The model specification and feature set are held constant. Only the validation
design changes.

The random split allows pages from the same client to occur in both training
and test data. The grouped split assigns each client entirely to one side,
which better measures generalization to clients the model has not seen.

In [4]:
FEATURES = [
    "log_impressions_early7",
    "log_impressions_recent7",
    "recent_trend_pct",
    "ctr_early7_pct",
    "ctr_recent7_pct",
    "avg_position_early7",
    "avg_position_recent7",
    "position_change"
]

TARGET = "is_declining_next15d"


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def make_logreg():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ])

In [7]:
import numpy as np

In [8]:
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

row_ids = np.arange(len(df))

random_train_idx, random_test_idx = train_test_split(
    row_ids,
    test_size=0.20,
    random_state=42,
    stratify=df[TARGET]
)

random_train = df.iloc[random_train_idx].copy()
random_test = df.iloc[random_test_idx].copy()

random_client_overlap = len(
    set(random_train["client_hash_id"])
    &
    set(random_test["client_hash_id"])
)

print("Random train rows:", len(random_train))
print("Random test rows:", len(random_test))
print("Random test base rate:", round(random_test[TARGET].mean(), 3))
print("Clients present in BOTH sets:", random_client_overlap)

NameError: name 'df' is not defined

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.